# File activity

Create, read, rename and delete a file, then inspect its history in Trail.

Start JupyterLab and Trail as in [setup](setup.md), then open this notebook in `trail-tutorial`.

These examples use file-close notifications available on Linux. On macOS and Windows, records can take longer to appear and reads may not be reported.

Event examples are from earlier runs. Ids, times and notebook or map names will differ.


## Create a CSV

In [ ]:
from pathlib import Path

path = Path('data/counts.csv')
path.write_text('name,count\na,1\nb,2\n')


Trail tracks the new file and records its write as a `FileAccessEvent` with `event_type: created`. It groups the operating system's notifications for a write into one record.

```text
5. FileAccessEvent
    id: edd99d90fdd94d418206631ddf1aa01e
    timestamp: 16:29:08.334
    src_path: data/counts.csv
    event_type: created
    started: 16:29:08.333
    st_size: 19 B
    st_mtime: 2026-11-12 08:29:08
    entry: 84a415bb1f1d4c2c842add1d6d4fd9a4
```


## Change columns

In [ ]:
path.write_text('name,count,total\na,1,2\nb,2,4\n')


The write record has `event_type: modified` and lists the column change. Trail compares file structure, such as columns and dimensions; it does not compare individual data values.

The record's `started` and `timestamp` fields mark the activity's start and last write. `st_size` and `st_mtime` describe the file when the activity ended.

```text
7. FileAccessEvent
    id: 8ba9c4693c64450b97f1c225069a9a74
    timestamp: 16:29:11.811
    src_path: data/counts.csv
    event_type: modified
    started: 16:29:11.810
    st_size: 29 B
    st_mtime: 2026-11-12 08:29:11
    change: columns: ['name', 'count'] → ['name', 'count', 'total']
    entry: 84a415bb1f1d4c2c842add1d6d4fd9a4
```


## Replace a file

In [ ]:
import os

temporary = Path('data/.counts.csv.tmp')
temporary.write_text('name,count\na,1\nb,2\n')
os.replace(temporary, path)


Some programs save to a temporary file, then replace the original. Trail records a modification to `counts.csv`. The temporary file does not match the tracking rules.

```text
9. FileAccessEvent
    id: 148845137c9047b084e184170d82a86f
    timestamp: 16:29:15.283
    src_path: data/counts.csv
    event_type: modified
    started: 16:29:15.283
    st_size: 19 B
    st_mtime: 2026-11-12 08:29:15
    change: columns: ['name', 'count', 'total'] → ['name', 'count']
    entry: 84a415bb1f1d4c2c842add1d6d4fd9a4
```


## Read a file

In [ ]:
path.read_text()


On Linux, this produces an `opened` record. Trail excludes its own reads from the log.

```text
11. FileAccessEvent
    id: 1e072c7f7d684c8cb439c9be03ad6ef5
    timestamp: 16:29:18.769
    src_path: data/counts.csv
    event_type: opened
    started: 16:29:18.769
    st_size: 19 B
    st_mtime: 2026-11-12 08:29:15
    entry: 84a415bb1f1d4c2c842add1d6d4fd9a4
```


## Rename a file

In [ ]:
path = path.rename('data/totals.csv')


The move record includes the old and new paths.

```text
12. WatchdogEvent
    id: 77564b9ab7324ede9206e1615aa7a6b3
    timestamp: 16:29:22.375
    src_path: data/counts.csv
    dest_path: data/totals.csv
    event_type: moved
    entry: 84a415bb1f1d4c2c842add1d6d4fd9a4
```


## Delete a file

In [ ]:
path.unlink()


The deletion keeps the same asset id as the earlier writes and rename.

```text
14. WatchdogEvent
    id: 5907dc1d8a23468eace1a4e4efa9cbf5
    timestamp: 16:29:25.854
    src_path: data/totals.csv
    event_type: deleted
    entry: 84a415bb1f1d4c2c842add1d6d4fd9a4
```

The rename produces a `WatchdogEvent` with `event_type: moved` and both paths. The deletion produces `event_type: deleted`. The records share an `entry` id, so the file's history survives the rename.


## Create a GeoPackage

This example needs GeoPandas. The next cell installs it into the notebook's environment if it is missing.

In [ ]:
from importlib.util import find_spec

if find_spec('geopandas') is None:
    %pip install geopandas

In [ ]:
import geopandas as gpd

crossings = gpd.read_file('data/crossings.geojson')
crossings.to_file('data/crossings.gpkg', driver='GPKG')


Trail tracks `data/crossings.gpkg` and records its creation. The next example changes this file.

## Change a GeoPackage

In [ ]:
(
    crossings
    .to_crs(3857)
    .assign(lanes=2)
    .to_file('data/crossings.gpkg', driver='GPKG')
)


The write record lists the CRS, bounds and column changes. SQLite journal activity is grouped with the GeoPackage's write.

```text
20. FileAccessEvent
    id: 1ea29be9c98f45a79603c4d7418b06d8
    timestamp: 16:29:32.916
    src_path: data/crossings.gpkg
    event_type: modified
    started: 16:29:32.782
    st_size: 96.0 KiB
    st_mtime: 2026-11-12 08:29:32
    change: layer 'crossings' CRS EPSG:4326 → EPSG:3857
    change: layers.crossings.bounds: [-122.27, 37.87, -122.26, 37.87] → [-13611034.13929356, 4561077.415263828, -13609920.9443856...
    change: column 'lanes' added in layer 'crossings'
    entry: 17e63571805c43d598d134fc00ac3bd5
```


## Startup changes

Enter `quit` in Trail before running this cell.

In [ ]:
(
    crossings
    .assign(lanes=2, checked=True)
    .to_file('data/crossings.gpkg', driver='GPKG')
)


Start `trail` again from the same project. It compares tracked files with their saved state and records changes as a `MetadataEvent` with `basis: startup`.

This records the difference found at startup, not every edit made while Trail was stopped.

```text
42. MetadataEvent
    id: dd056d740bef4eada2f6bd5dc4ce011d
    timestamp: 16:30:09.809
    src_path: data/crossings.gpkg
    format: geopackage
    basis: startup
    st_size: 96.0 KiB
    st_mtime: 2026-11-12 08:30:09
    change: column 'checked' added in layer 'crossings'
    entry: 17e63571805c43d598d134fc00ac3bd5
```

## Other formats

| Format | What Trail compares |
|---|---|
| CSV, TSV | Delimiter and columns. |
| GeoJSON | Feature count and properties. |
| GeoPackage, FlatGeobuf, Shapefile | Features, columns, CRS and other format-specific structure. |
| GeoTIFF | Dimensions, bands, CRS and nodata. |
| Parquet, GeoParquet, Feather, Arrow | Columns, available row counts and geometry CRS. |
| NumPy | Arrays, shapes and data types. |
| Zarr | Arrays, shapes, chunks and data types. |
| NetCDF, HDF5 | Dimensions, variables or datasets. |
| QGIS, JupyterGIS | Layers, sources, layer order and project settings. |

Parquet and Arrow formats, NetCDF, and HDF5 need extra readers. From the `trail-tutorial` folder, install them with:

```bash
python -m pip install -e '../trail[formats]'
```

Without a reader, Trail records file activity without structural changes. Structural comparison also skips GeoJSON files over 32 MiB and map projects over 64 MiB.

Trail treats a Zarr folder as one asset and groups shapefile sidecars with the `.shp`. Programs that delete and rewrite files can produce both deletion and creation records. Zarr changes may appear later in a separate `MetadataEvent`.

## Query file history

Run these in Trail:

```text
events src_path=data/crossings.gpkg
events cls=FileAccessEvent event_type=modified
events cls=FileAccessEvent event_type=opened
events cls=MetadataEvent basis=startup
```

Trail does not identify the program that wrote a file. [Notebook execution](notebook.ipynb) shows how cell times can help relate a write to a run.
